# Machine Learning Assignment 2 - Model Training & Evaluation
**Degree**: M.Tech (AIML) - BITS Pilani (WILP)  
**Course**: Machine Learning  
**Topic**: Comparative Evaluation of 5 Classification Algorithms on Breast Cancer Wisconsin (Diagnostic) Dataset  

---
### Assignment Objectives:
1. Select a public classification dataset with >= 12 features and >= 500 instances.
2. Implement and evaluate 5 core classification models:
   - Logistic Regression
   - Decision Tree Classifier
   - K-Nearest Neighbors Classifier (KNN)
   - Naive Bayes Classifier (GaussianNB)
   - Support Vector Machine (SVM)
3. Compute 6 mandatory evaluation metrics:
   - Accuracy, AUC, Precision, Recall, F1 Score, Matthews Correlation Coefficient (MCC)
4. Save trained models and preprocessors for deployment in a Streamlit web application.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

# 1. Load dataset
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

# Align target: 1 = Malignant (positive), 0 = Benign (negative)
df['target'] = 1 - df['target']
print('Dataset dimensions:', df.shape)
print('Missing values:', df.isnull().sum().sum())
print('Class distribution:\n', df['target'].value_counts())
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['target'])
y = df['target']

# 80-20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Decision Tree': Pipeline([
        ('classifier', DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_split=4))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Naive Bayes': Pipeline([
        ('classifier', GaussianNB())
    ]),
    'Support Vector Machine': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='rbf', probability=True, random_state=42))
    ])
}

In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

results = []
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    results.append({
        'ML Model Name': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_proba), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1': round(f1_score(y_test, y_pred), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
import joblib

joblib.dump(models['Logistic Regression'], 'logistic_regression.joblib')
joblib.dump(models['Decision Tree'], 'decision_tree.joblib')
joblib.dump(models['KNN'], 'knn.joblib')
joblib.dump(models['Naive Bayes'], 'naive_bayes.joblib')
joblib.dump(models['Support Vector Machine'], 'svm.joblib')
print('Saved all 5 models successfully.')